In [35]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EvalPrediction,
)
from peft import PrefixTuningConfig, get_peft_model, TaskType
import evaluate

# Load dataset

In [36]:

dataset = load_dataset("stanfordnlp/sst2")

# Safe sub-sampling
train_size = min(5000, len(dataset["train"]))
val_size = min(len(dataset["validation"]), 872)

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(train_size))
dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(val_size))

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Train size: 5000
Validation size: 872


# Tokenizer + model

In [37]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12592.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

# Tokenization

In [38]:

def preprocess(example):
    return tokenizer(example["sentence"], truncation=True)

encoded = dataset.map(preprocess, batched=True)

# Keep only needed columns
keep_cols = ["input_ids", "attention_mask", "label"]
encoded = encoded.remove_columns(
    [c for c in encoded["train"].column_names if c not in keep_cols]
)

encoded = encoded.rename_column("label", "labels")
encoded.set_format("torch")

# Prefix Tuning

In [39]:
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20
)

model = get_peft_model(model, prefix_config)

model.print_trainable_parameters()

trainable params: 370,178 || all params: 109,853,956 || trainable%: 0.3370


# Metrics

In [40]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# Training args

In [41]:
training_args = TrainingArguments(
    output_dir="./prefix_tuning_sst2_final",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Trainer

In [42]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train

In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.643037,0.616571,0.627294
2,0.560919,0.525503,0.724771


TrainOutput(global_step=626, training_loss=0.6261441623821807, metrics={'train_runtime': 16.6536, 'train_samples_per_second': 600.471, 'train_steps_per_second': 37.59, 'total_flos': 178577502187968.0, 'train_loss': 0.6261441623821807, 'epoch': 2.0})

# Save the model

In [44]:
model.save_pretrained("./prefix_tuned_model_final")
tokenizer.save_pretrained("./prefix_tuned_model_final")

print("Training complete ✅")

Training complete ✅


# Inference on single 

In [45]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# Load model + adapter
base_model_name = "bert-base-uncased"
adapter_path = "./prefix_tuned_model_final"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# ---- single sentence ----
text = "This movie was really good and exciting."

inputs = tokenizer(text, return_tensors="pt", truncation=True)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)

pred = torch.argmax(probs, dim=-1).item()

label = "positive" if pred == 1 else "negative"

print("Text:", text)
print("Prediction:", label)
print("Confidence:", float(probs[0][pred]))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11547.68it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Text: This movie was really good and exciting.
Prediction: positive
Confidence: 0.8958657383918762


# Multiple inference

In [46]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# =========================
# 1. Load base model + adapter
# =========================
base_model_name = "bert-base-uncased"
adapter_path = "./prefix_tuned_model_final"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2
)

model = PeftModel.from_pretrained(base_model, adapter_path)

model.eval()

# =========================
# 2. Prediction function
# =========================
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)

    pred = torch.argmax(probs, dim=-1).item()

    label = "positive" if pred == 1 else "negative"

    return {
        "text": text,
        "prediction": label,
        "confidence": float(probs[0][pred])
    }

# =========================
# 3. Test examples
# =========================
examples = [
    "This movie was amazing and full of emotion.",
    "I hated every minute of it.",
    "It was okay, not great but not bad either.",
]

for ex in examples:
    print(predict(ex))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14068.68it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

{'text': 'This movie was amazing and full of emotion.', 'prediction': 'positive', 'confidence': 0.9266737103462219}
{'text': 'I hated every minute of it.', 'prediction': 'negative', 'confidence': 0.8715918660163879}
{'text': 'It was okay, not great but not bad either.', 'prediction': 'negative', 'confidence': 0.5554060935974121}
